# 📈 Linear Regression from Scratch

This notebook walks through:
1. Simple Linear Regression (Gradient Descent vs Normal Equation)
2. Multiple Linear Regression
3. Regularization: L1 (Lasso) & L2 (Ridge)
4. Polynomial Regression
5. Full Diagnostic Report

**No scikit-learn** — everything is built from scratch using only NumPy.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from linear_regression import (
    LinearRegression,
    StandardScaler, MinMaxScaler, PolynomialFeatures,
    train_test_split,
    mean_squared_error, mean_absolute_error,
    root_mean_squared_error, r2_score, adjusted_r2_score,
    plot_regression_line, plot_residuals, plot_cost_history,
    plot_feature_importance, plot_actual_vs_predicted, plot_full_report,
)

print('✅ Imports successful')

---
## 1. Simple Linear Regression

We'll generate data from: **y = 3x + 5 + noise**

In [ ]:
rng = np.random.default_rng(42)
X = rng.uniform(-5, 5, size=(150, 1))
y = 3 * X.flatten() + 5 + rng.normal(0, 1.5, 150)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Train: {len(X_train)} samples | Test: {len(X_test)} samples')

### 1a. Gradient Descent

In [ ]:
gd = LinearRegression(solver='gradient_descent', learning_rate=0.1, n_iterations=500)
gd.fit(X_train_s, y_train)

y_pred = gd.predict(X_test_s)
print(f'Weight : {gd.weights_[0]:.4f}  (true: ~3)')
print(f'Bias   : {gd.bias_:.4f}  (true: ~5)')
print(f'R²     : {r2_score(y_test, y_pred):.4f}')
print(f'RMSE   : {root_mean_squared_error(y_test, y_pred):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_regression_line(X_test_s, y_test, gd.predict(X_test_s), ax=axes[0], title='GD: Regression Fit')
plot_cost_history(gd.cost_history_, ax=axes[1])
plt.tight_layout()
plt.show()

### 1b. Normal Equation (closed-form solution)

In [ ]:
ne = LinearRegression(solver='normal_equation')
ne.fit(X_train_s, y_train)

y_pred_ne = ne.predict(X_test_s)
print(f'Weight : {ne.weights_[0]:.4f}')
print(f'Bias   : {ne.bias_:.4f}')
print(f'R²     : {r2_score(y_test, y_pred_ne):.4f}')

---
## 2. Multiple Linear Regression

In [ ]:
rng2 = np.random.default_rng(0)
X_m = rng2.standard_normal((300, 4))
true_w = np.array([4.0, -2.5, 1.8, 3.2])
y_m = X_m @ true_w + 7.0 + rng2.normal(0, 1, 300)

X_tr, X_te, y_tr, y_te = train_test_split(X_m, y_m, test_size=0.2, random_state=5)
sc2 = StandardScaler()
X_tr_s = sc2.fit_transform(X_tr)
X_te_s = sc2.transform(X_te)

model_m = LinearRegression(solver='gradient_descent', n_iterations=2000, learning_rate=0.1)
model_m.fit(X_tr_s, y_tr)

print(f'True weights     : {true_w}')
print(f'Learned weights  : {np.round(model_m.weights_, 3)}')
print(f'Bias             : {model_m.bias_:.4f}  (true: 7.0)')
print(f'R² (test)        : {model_m.score(X_te_s, y_te):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_actual_vs_predicted(y_te, model_m.predict(X_te_s), ax=axes[0])
plot_feature_importance(model_m.weights_, feature_names=['x1','x2','x3','x4'], ax=axes[1])
plt.tight_layout()
plt.show()

---
## 3. Regularization Comparison: No Reg vs L2 vs L1

In [ ]:
configs = [
    ('No Reg',    dict(regularization=None)),
    ('L2 λ=0.5', dict(regularization='l2', lambda_=0.5)),
    ('L1 λ=0.1', dict(regularization='l1', lambda_=0.1)),
]

print(f'{"Config":<14} {"R² Train":>10} {"R² Test":>10} {"MSE":>10}')
print('-' * 46)
for name, kwargs in configs:
    m = LinearRegression(solver='gradient_descent', n_iterations=2000, learning_rate=0.1, **kwargs)
    m.fit(X_tr_s, y_tr)
    r2_tr = m.score(X_tr_s, y_tr)
    r2_te = m.score(X_te_s, y_te)
    mse   = mean_squared_error(y_te, m.predict(X_te_s))
    print(f'{name:<14} {r2_tr:>10.4f} {r2_te:>10.4f} {mse:>10.4f}')

---
## 4. Polynomial Regression

Fitting a cubic curve: **y = 0.5x³ - x² + 2x + 3**

In [ ]:
rng3 = np.random.default_rng(7)
X_p = rng3.uniform(-4, 4, (200, 1))
y_p = 0.5*X_p.flatten()**3 - X_p.flatten()**2 + 2*X_p.flatten() + 3 + rng3.normal(0, 2, 200)

X_ptr, X_pte, y_ptr, y_pte = train_test_split(X_p, y_p, test_size=0.2, random_state=0)

print(f'{"Degree":<8} {"Train R²":>10} {"Test R²":>10} {"MSE":>10}')
print('-' * 40)

best_model, best_scaler, best_pf = None, None, None

for deg in [1, 2, 3, 4, 5]:
    pf   = PolynomialFeatures(degree=deg, include_bias=False)
    Xtr  = pf.fit_transform(X_ptr)
    Xte  = pf.fit_transform(X_pte)
    sc   = StandardScaler()
    Xtr  = sc.fit_transform(Xtr)
    Xte  = sc.transform(Xte)
    m    = LinearRegression(solver='normal_equation', regularization='l2', lambda_=0.001)
    m.fit(Xtr, y_ptr)
    r2tr = r2_score(y_ptr, m.predict(Xtr))
    r2te = r2_score(y_pte, m.predict(Xte))
    mse  = mean_squared_error(y_pte, m.predict(Xte))
    print(f'{deg:<8} {r2tr:>10.4f} {r2te:>10.4f} {mse:>10.4f}')
    if deg == 3:
        best_model, best_scaler, best_pf = m, sc, pf

In [ ]:
X_line = np.linspace(-4, 4, 300).reshape(-1, 1)
X_line_poly = best_pf.fit_transform(X_line)
X_line_s    = best_scaler.transform(X_line_poly)
y_line      = best_model.predict(X_line_s)

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(X_ptr.flatten(), y_ptr, s=20, alpha=0.5, color='#2563EB', label='Train')
ax.scatter(X_pte.flatten(), y_pte, s=20, alpha=0.7, color='#10B981', label='Test')
ax.plot(X_line.flatten(), y_line, color='#F59E0B', lw=2.5, label='Degree-3 fit')
ax.set_title('Polynomial Regression — Degree 3', fontweight='bold')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

---
## 5. Full Diagnostic Report

A 2×2 dashboard: Cost history, Actual vs Predicted, Residuals, Feature Importance.

In [ ]:
fig = plot_full_report(
    model_m, X_tr_s, y_tr, X_te_s, y_te,
    feature_names=['x1', 'x2', 'x3', 'x4']
)
plt.show()

---
## 6. All Metrics Demo

In [ ]:
y_true = y_te
y_pred = model_m.predict(X_te_s)
n_feat = model_m.n_features_

print(f'MSE          : {mean_squared_error(y_true, y_pred):.4f}')
print(f'MAE          : {mean_absolute_error(y_true, y_pred):.4f}')
print(f'RMSE         : {root_mean_squared_error(y_true, y_pred):.4f}')
print(f'R²           : {r2_score(y_true, y_pred):.4f}')
print(f'Adjusted R²  : {adjusted_r2_score(y_true, y_pred, n_feat):.4f}')